In [2]:
from model.Model import model_MSG
# from model.Plotter import plotter_MSG
# from model.coders_linkage import CODERS_pull
from pstats import SortKey, Stats

import os
import sys
import pandas as pd
# Autoreload modules when changes are applied to them
%load_ext autoreload 
%autoreload all


In [3]:

# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

%reload_ext autoreload
# Set-up
# Sys and OS Operations
os.chdir("../../")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
msg = model_MSG(scenario_name='baseline')

In [4]:

platform, msg, msg_folder = msg.setup_scenario()

2025-02-25 10:46:04,151  INFO at.ac.iiasa.ixmp.Platform:165 - Welcome to the IX modeling platform!
2025-02-25 10:46:04,157  INFO at.ac.iiasa.ixmp.Platform:166 -  connected to database 'jdbc:hsqldb:file:/Users/muhammadawais/.local/share/ixmp/localdb/default' (user: ixmp)...

Model Logs: Setting up scenario


		Model creation time: 8 Min and 57 Sec 		
Solving the model...

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 02/25/25 10:54:33 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225a/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgD

2025-02-25 10:55:15,052 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-02-25 10:55:15,058 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!



				Solving process time: 1 Min and 16 Sec				
processing Table: Resource|Extraction
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
No unit conversion factor found to convert GWa to PJ/yr
processing Table: Resource|Cumulative Extraction
No unit conversion factor found to convert GWa to ZJ
No unit conversion factor fo

In [231]:
msg.remove_solution()

In [240]:
df = msg.par('relation_activity')

In [264]:
df = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/baseline/model_default_v19.xlsx',sheet_name = 'relation_activity')

In [265]:
list = ['TCE_Emission', 'TCE_Emission_NAM_PAO_EEU_WEU',
       'TCE_Emission_NAM_PAO_EEU_WEU_FSU', 'TCF4_Emission',
       'TCH4_Emission', 'THFC_Emission', 'global_GCO2', 'TN2O_Emission','TSF6_Emission']

In [266]:
df_rel = df[df['relation'].isin(list)]

In [267]:
df_rel_add = df[~df['relation'].isin(list)]

In [282]:
df_rel_add

,relation,node_rel,year_rel,node_loc,technology,year_act,mode,value,unit
495,elec_relations2_renewable,Alberta,2010,Alberta,bio_istig,2010,M1,1.000000,GWa/a
496,elec_relations2_renewable,Alberta,2015,Alberta,bio_istig,2015,M1,1.000000,GWa/a
497,elec_relations2_renewable,Alberta,2020,Alberta,bio_istig,2020,M1,1.000000,GWa/a
498,elec_relations2_renewable,Alberta,2025,Alberta,bio_istig,2025,M1,1.000000,GWa/a
499,elec_relations2_renewable,Alberta,2030,Alberta,bio_istig,2030,M1,1.000000,GWa/a
...,...,...,...,...,...,...,...,...,...
284467,res_marg,Yukon,2040,Yukon,elec_t_d,2040,M1,-2.513060,GWa
284468,res_marg,Yukon,2045,Yukon,elec_t_d,2045,M1,-2.511947,GWa
284469,res_marg,Yukon,2050,Yukon,elec_t_d,2050,M1,-2.514295,GWa
284470,res_marg,Yukon,2055,Yukon,elec_t_d,2055,M1,-2.516617,GWa


In [268]:
df_rel['node_rel'] = 'Canada'

In [291]:
df = msg.par('relation_activity')

In [269]:
rels_remove = ['TCE_Emission_NAM_PAO_EEU_WEU','TCE_Emission_NAM_PAO_EEU_WEU_FSU','CO2_Emission_Global_Total','TCE_regional']

In [334]:
from model.utlities.utilities_main.message_utils import add_bound_from_hist

In [344]:
tecs= ['gas_exp','oil_exp']

In [349]:
bound_lo_yr = 0

In [347]:
yrs = [int(x) for x in msg.set("year").tolist() if int(x) >= msg.firstmodelyear]
yr_hist = max([x for x in msg.set("year") if x < msg.firstmodelyear])

# Loading historical data
df_hist = msg.par(
    "historical_activity",
    {"node_loc": nodes, "technology": tecs, "year_act": yr_hist},
)

In [351]:
msg.check_out()

In [352]:
df_bdl = pd.DataFrame(columns=msg.par("bound_activity_lo").columns)
for yr in yrs:
    d = df_hist.copy()
    d["year_act"] = yr
    d["value"] *= 1 + (yr - yr_hist) * bound_lo_yr
    df_bdl = pd.concat([df_bdl, d], ignore_index=True)
df_bdl.loc[df_bdl["value"] < 0, "value"] = 0
msg.add_par("bound_activity_lo", df_bdl)
print(
    '- Lower bound with grwoth of "{}" per year addedr'
    ' in "{}".'.format(bound_lo_yr, nodes)
)


- Lower bound with grwoth of "0" per year addedr in "['Alberta', 'BritishColumbia', 'Manitoba', 'NewBrunswick', 'NewfoundlandandLabrador', 'NorthwestTerritories', 'NovaScotia', 'Nunavut', 'Ontario', 'PrinceEdwardIsland', 'Quebec', 'Saskatchewan', 'Yukon']".


In [355]:
remove_og_bound(msg)

In [357]:
df_glo = msg.par('growth_activity_lo')
df_gup = msg.par('growth_activity_up')
df_iup = msg.par('initial_activity_up')
df_ilow = msg.par('initial_activity_lo')


import pandas as pd

# Define the filter function
def remove_extraction_rows(df):
    return df[df.apply(lambda row: row.astype(str).str.contains('gas_exp|oil_exp', regex=True)).any(axis=1)]

# Apply the function to each dataframe
df_glo = remove_extraction_rows(df_glo)
df_gup = remove_extraction_rows(df_gup)
df_iup = remove_extraction_rows(df_iup)
df_ilow = remove_extraction_rows(df_ilow)


In [359]:
with msg.transact(''):
    msg.remove_par('growth_activity_lo', df_glo)
    msg.remove_par('growth_activity_up', df_gup)
    msg.remove_par('initial_activity_up', df_iup)
    msg.remove_par('initial_activity_lo', df_ilow)

In [340]:
nodes = msg.set('node').tolist()[2:]


In [345]:
add_bound_from_hist(msg,tecs, nodes, bound_lo_yr = 0.0001)

>>>Warning<<<: No historical activity for technologies "['Alberta', 'BritishColumbia', 'Manitoba', 'NewBrunswick', 'NewfoundlandandLabrador', 'NorthwestTerritories', 'NovaScotia', 'Nunavut', 'Ontario', 'PrinceEdwardIsland', 'Quebec', 'Saskatchewan', 'Yukon']" in "['gas_exp', 'oil_exp']", no bounds added.


In [ ]:
df = msg.par('bound_actio')

In [342]:
msg.remove_solution()

In [298]:
transport_update(msg)

- coal_trp removed.
- Capex removed and vintaging corrected for ['h2_fc_trp']
- O&M updated for transport technologies.


In [330]:
df = msg.set('cat_emission')

In [327]:
df_remove = df[~df['emission'].isin(emis)]

In [383]:
import pandas as pd
import pandas as pd

# Define the full dataset exactly as provided
data = ["Ontario", "gas_exp", 2020, "M1", "year", 4.65615e-12]
    



# Convert into DataFrame
df_condition = pd.DataFrame(data, columns=['node_loc', 'technology', 'year_act','mode' ,'time', 'value'])

# Load another target dataset from which values will be subtracted (Example data)
data_target = msg.par('bound_activity_lo',filters = {'technology': ['gas_exp','oil_exp']})

df_target = data_target

#
# Filter df_target based on matching 'node_loc' and 'year' from df_condition
df_target_filtered = df_target[
    df_target[['node_loc', 'year_act']].apply(tuple, axis=1).isin(
        df_condition[['node_loc', 'year_act']].apply(tuple, axis=1)
    )
]

In [389]:

# Load another target dataset from which values will be subtracted (Example data)
data_target = msg.par('bound_activity_lo',filters = {'technology': ['gas_exp']})

In [391]:
data_target.loc[(data_target['node_loc'] == 'Ontario') & (data_target['year_act'] == 2020), 'value'] -=4.65615e-12

In [392]:
with msg.transact():
    msg.add_par('bound_activity_lo', data_target)

In [384]:
# Merge the filtered target dataset with the condition dataset
df_merged = df_target_filtered.merge(df_condition, on=['node_loc', 'technology', 'year_act', 'mode', 'time'], how='inner', suffixes=('_target', '_condition'))


In [385]:

# Perform the subtraction
df_merged['value'] = df_merged['value_target'] - df_merged['value_condition']

# Keep only relevant columns
df_final = df_merged[['node_loc', 'technology', 'year_act', 'mode', 'time', 'value']]



In [386]:
with msg.transact():
    msg.add_par('bound_activity_lo', df_final)

In [311]:
emis = ['TCE','TCE_CO2', 'TCE_non-CO2']

In [314]:
df_emis_maintain = df[df['emission'].isin(emis)]

In [315]:
df_emis_remove = df[~df['emission'].isin(emis)]

In [305]:
df = df[df['relation'].isin(['CO2_Emission_Global_Total'])]

df_Reob

In [437]:
import pandas as pd

# Use an alternative way to convert range to list
type_year_values = [year for year in range(2025, 2061, 5)]

# Manually create the DataFrame
df_bound_net = pd.DataFrame({
    "node": ["Canada"] * len(type_year_values),
    "type_emission": ["TCE"] * len(type_year_values),
    "type_tec": ["all"] * len(type_year_values),
    "type_year": type_year_values,
    "value": [200] * len(type_year_values)
})




     node type_emission type_tec  type_year  value
0  Canada           TCE      all       2025    200
1  Canada           TCE      all       2030    200
2  Canada           TCE      all       2035    200
3  Canada           TCE      all       2040    200
4  Canada           TCE      all       2045    200
5  Canada           TCE      all       2050    200
6  Canada           TCE      all       2055    200
7  Canada           TCE      all       2060    200


In [439]:
msg.remove_solution()

In [471]:
with msg.transact(''):
    # msg.remove_par('bound_emission', df)
    msg.add_par('bound_emission', df_upd)

In [473]:
df = msg.par('bound_emission')

In [474]:

from model.utlities.demands import adjust_demands

In [481]:
df = msg.par('demand')


In [ ]:
import pandas as pd

# Define cumulative rate of change
rate_of_change = {
    2025: 0.052566283,
    2030: 0.049920952,
    2035: 0.045423593,
    2040: 0.044337335,
    2045: 0.040088765,
    2050: 0.038095737,
    2055: 0.038278554,
    2060: 0.03846426
}

# Convert year column to integer
df["year"] = df["year"].astype(int)

# Get the 2020 base values for each (node, commodity) pair
base_values = df[df["year"] == 2020].set_index(["node", "commodity"])["value"]

# Sort DataFrame to ensure chronological updates
df = df.sort_values(by=["node", "commodity", "year"])

# Apply cumulative rate of change iteratively
for year in sorted(rate_of_change.keys()):
    mask = df["year"] == year
    previous_year = year - 5  # Assuming a 5-year interval between records
    prev_mask = df["year"] == previous_year

    df.loc[mask, "value"] = df.loc[mask].apply(
        lambda row: df.loc[
            (df["node"] == row["node"]) & (df["commodity"] == row["commodity"]) & (df["year"] == previous_year), "value"
        ].values[0] * (1 + rate_of_change[year])
        if not df.loc[
            (df["node"] == row["node"]) & (df["commodity"] == row["commodity"]) & (df["year"] == previous_year), "value"
        ].empty else base_values.get((row["node"], row["commodity"]), 0) * (1 + rate_of_change[year]),
        axis=1
    )


In [487]:
with msg.transact(''):
    msg.add_par('demand', df)

In [486]:
(5.715158- 5.443418)/5.443418

0.049920840177991

In [467]:
df_upd = df[df['type_year'] != 'cumulative']

In [468]:
df_upd['value'] = 300

In [ ]:
df['value'] = df['value'] 

In [505]:
df = msg.par('bound_emission')

In [506]:
df.loc[df['type_year']== '2020','value'] = 350

In [500]:
df['type_year'] ='2020'

In [502]:
msg.remove_solution()

In [507]:
with msg.transact(''):
    msg.add_par('bound_emission', df)

In [508]:
caseName = msg.model + '__' + msg.scenario + '__v' + str(msg.version)

msg.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4', "scaind": "1"})

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 02/27/25 23:38:07 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225i/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__baseline__v65.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__baseline__v65.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__baseline__v65.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
  

2025-02-27 23:43:05,039 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-02-27 23:43:05,050 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [536]:
netzero_1.to_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/model/scenarios/netzero/model_default_netzero_v7.xlsx')

 sec. (1175324.34 ticks, 309425 iterations)
Iteration: 309555   Dual objective     =    3457690166.390197
Elapsed time = 4136.21 sec. (1186548.37 ticks, 309556 iterations)
Iteration: 309667   Dual objective     =    3457928263.645926
Elapsed time = 4455.16 sec. (1196584.92 ticks, 309668 iterations)
Elapsed time = 4515.14 sec. (1208564.23 ticks, 309688 iterations)
Iteration: 309771   Dual objective     =    3458039476.797681
Elapsed time = 4635.06 sec. (1227443.55 ticks, 309783 iterations)
Elapsed time = 4717.51 sec. (1245325.65 ticks, 310055 iterations)
Iteration: 310055   Dual objective     =    3458484982.742517
Elapsed time = 4766.88 sec. (1255330.93 ticks, 310415 iterations)
Iteration: 310519   Dual objective     =    3460250974.888538
Elapsed time = 4801.45 sec. (1265806.78 ticks, 310520 iterations)
Elapsed time = 4835.02 sec. (1275840.27 ticks, 310584 iterations)
Iteration: 310856   Dual objective     =    3461800091.445383
Elapsed time = 4871.69 sec. (1287107.21 ticks, 310861 it

In [270]:
df_rel = df_rel[~(df_rel['relation'].isin(rels_remove))]

In [276]:
df_rel

,relation,node_rel,year_rel,node_loc,technology,year_act,mode,value,unit
0,TCE_Emission,Canada,2010,Alberta,CF4_TCE,2010,M1,2015.450,???
3,TCF4_Emission,Canada,2010,Alberta,CF4_TCE,2010,M1,2015.450,???
4,TCE_Emission,Canada,2015,Alberta,CF4_TCE,2015,M1,2015.450,???
7,TCF4_Emission,Canada,2015,Alberta,CF4_TCE,2015,M1,2015.450,???
8,TCE_Emission,Canada,2020,Alberta,CF4_TCE,2020,M1,2015.450,???
...,...,...,...,...,...,...,...,...,...
264694,global_GCO2,Canada,2040,Yukon,oil_exp,2040,M1,-0.631,GWa/a
264695,global_GCO2,Canada,2045,Yukon,oil_exp,2045,M1,-0.631,GWa/a
264696,global_GCO2,Canada,2050,Yukon,oil_exp,2050,M1,-0.631,GWa/a
264697,global_GCO2,Canada,2055,Yukon,oil_exp,2055,M1,-0.631,GWa/a


In [279]:
df = msg.par('relation_activity')

In [280]:
 #Given net-zero emission trajectory from Environment Canada
data = {
    "type_year": list(range(2025, 2061, 5)),
    "value": [659, 549, 439, 329, 220, 110, 110,50,50]
}

# Create DataFrame
df_bound_net = pd.DataFrame(data)

# Add requi0.0116 * 1e3 *365 /8760red columns
df_bound_net.insert(0, "node", "Canada")
df_bound_net.insert(1, "type_emission", "TCE")
df_bound_net.insert(2, "type_tec", "all")

,relation,node_rel,year_rel,node_loc,technology,year_act,mode,value,unit
0,TCE_Emission,Canada,2010,Alberta,CF4_TCE,2010,M1,2015.450,???
1,TCF4_Emission,Canada,2010,Alberta,CF4_TCE,2010,M1,2015.450,???
2,TCE_Emission,Canada,2015,Alberta,CF4_TCE,2015,M1,2015.450,???
3,TCF4_Emission,Canada,2015,Alberta,CF4_TCE,2015,M1,2015.450,???
4,TCE_Emission,Canada,2020,Alberta,CF4_TCE,2020,M1,2015.450,???
...,...,...,...,...,...,...,...,...,...
8120,global_GCO2,Canada,2040,Yukon,oil_exp,2040,M1,-0.631,GWa/a
8121,global_GCO2,Canada,2045,Yukon,oil_exp,2045,M1,-0.631,GWa/a
8122,global_GCO2,Canada,2050,Yukon,oil_exp,2050,M1,-0.631,GWa/a
8123,global_GCO2,Canada,2055,Yukon,oil_exp,2055,M1,-0.631,GWa/a


In [283]:
msg.remove_solution()

In [284]:
with msg.transact(''):
    msg.add_par('relation_activity', df_rel_add)

In [124]:
netzero.version

19

In [510]:
netzero = msg.clone(msg.model,'netzero',keep_solution=False)


In [129]:
df = msg.par('tax_emission')

In [511]:
def carbon_price(msg):
    """Increase the carbon price to understand level of emissions reduction."""
    
    with msg.transact():
        df = msg.par('tax_emission')
        df['type_tec'] = 'all'
        df['value'] *=  500
        msg.add_par("tax_emission", df)

def remove_og_bound(msg):
    """Increase the carbon price to understand level of emissions reduction"""
    
    with msg.transact():
        df = msg.par('bound_activity_lo')
        df_remove = df[df['technology'].str.contains('oil_extr|gas_extr')]
        msg.remove_par("bound_activity_lo", df_remove)



In [417]:


def transfer_demands(base, scen):
    """Copies a demand from one scenario to another.

    Copies the demand ('variable') from one scenario
    to another and adds it as a 'parameter'.

    Parameters
    ----------
    base : :class:`message_ix.Scenario`
        source scenario from which demands should be copied

    scen : :class:`message_ix.Scenario`
        target scenario to which demands should be copied
    """
    print(f"Transfer data from {base.url} variable 'DEMAND'")
    print(f"to {scen.url} parameter 'demand'")

    df = base.var("DEMAND").rename(columns={"lvl": "value"}).assign(unit="GWa")
    scen.check_out()
    scen.add_par("demand", df)
    scen.commit("parameter demand updated from source variable")

    print(f"{len(df)} observations")

In [431]:
netzero.remove_solution()

In [512]:
carbon_price(netzero)


In [513]:
df = netzero.par('tax_emission')

In [514]:
df

,node,type_emission,type_tec,type_year,value,unit
0,Alberta,TCE,enduse_energy_tecs,2025,165.425062,???
1,Alberta,TCE,enduse_energy_tecs,2030,285.215625,???
2,Alberta,TCE,enduse_energy_tecs,2035,387.893250,???
3,Alberta,TCE,enduse_energy_tecs,2040,387.893250,???
4,Alberta,TCE,enduse_energy_tecs,2045,387.893250,???
...,...,...,...,...,...,...
203,Yukon,TCE,all,2040,193946.625000,???
204,Yukon,TCE,all,2045,193946.625000,???
205,Yukon,TCE,all,2050,193946.625000,???
206,Yukon,TCE,all,2055,193946.625000,???


In [429]:
df.loc[(df['type_tec'] == 'all'),'value'] *= 10

In [432]:
with netzero.transact():
    netzero.add_par('tax_emission', df)

In [515]:
caseName = netzero.model + '__' + netzero.scenario + '__v' + str(netzero.version)
netzero.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '2', "scaind": "1"})

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 02/28/25 00:11:27 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225i/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__netzero__v21.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__netzero__v21.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__netzero__v21.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
     

2025-02-28 00:13:23,214 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-02-28 00:13:23,233 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [516]:
netzero.var('EMISS', {'node': 'Canada', 'emission': 'TCE', 'type_tec': 'all'})

,node,emission,type_tec,year,lvl,mrg
0,Canada,TCE,all,2020,350.000000,0.0
1,Canada,TCE,all,2025,199.523825,0.0
2,Canada,TCE,all,2030,107.426983,0.0
3,Canada,TCE,all,2035,71.188808,0.0
4,Canada,TCE,all,2040,45.701622,0.0
5,Canada,TCE,all,2045,25.517094,0.0
6,Canada,TCE,all,2050,12.906891,0.0
7,Canada,TCE,all,2055,41.145122,0.0
8,Canada,TCE,all,2060,7.760384,0.0


In [399]:
from model.report.legacy.iamc_report_hackathon import report


In [537]:
scens=[msg,netzero_1]

In [540]:
report(mp= mp, scen=msg, out_dir="reporting_outputs")

processing Table: Resource|Extraction
processing Table: Resource|Cumulative Extraction
processing Table: Primary Energy
PostProcess.out() got an unexpected keyword argument 'inpfilter'
processing Table: Primary Energy (substitution method)
PostProcess.out() got an unexpected keyword argument 'inpfilter'
processing Table: Final Energy
processing Table: Secondary Energy|Electricity
processing Table: Secondary Energy|Heat
processing Table: Secondary Energy
processing Table: Secondary Energy|Gases
processing Table: Secondary Energy|Solids
processing Table: Emissions|CO2
processing Table: Carbon Sequestration
processing Table: Emissions|BC
processing Table: Emissions|OC
processing Table: Emissions|CO
processing Table: Emissions|N2O
processing Table: Emissions|CH4
processing Table: Emissions|NH3
processing Table: Emissions|Sulfur
processing Table: Emissions|NOx
processing Table: Emissions|VOC
processing Table: Emissions|HFC
HFC125
No objects to concatenate
processing Table: Emissions
process

,2020,2025,2030,2035,2040,2045,2050,2055,2060,Model,Region,Scenario,Unit,Variable
1,8227.614414,9784.195663,8629.669358,10154.712721,10490.108882,15829.803904,15810.306795,62495.010094,62334.305256,MESSAGEix-CA,Alberta,baseline,PJ/yr,Resource|Extraction
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MESSAGEix-CA,Alberta,baseline,PJ/yr,Resource|Extraction|Coal
3,245.223044,610.358360,643.739674,856.669791,865.109657,875.840618,862.029728,160.704838,0.000000,MESSAGEix-CA,Alberta,baseline,PJ/yr,Resource|Extraction|Gas
4,245.223044,610.358360,643.739674,856.669791,865.109657,875.840618,862.029728,160.704838,0.000000,MESSAGEix-CA,Alberta,baseline,PJ/yr,Resource|Extraction|Gas|Conventional
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MESSAGEix-CA,Alberta,baseline,PJ/yr,Resource|Extraction|Gas|Unconventional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,1.411395,0.517471,0.530410,0.781063,1.366521,1.296196,3.489206,7.337030,-39.427097,MESSAGEix-CA,PrinceEdwardIsland,baseline,billion US$2010/yr,Cost|Cost Nodal Net
23,47.293558,20.606262,19.726275,21.783983,25.179567,25.752910,30.567362,38.534507,-41.882215,MESSAGEix-CA,Quebec,baseline,billion US$2010/yr,Cost|Cost Nodal Net
25,14.927321,4.530820,5.311469,5.023631,5.503950,6.712060,7.857478,17.812646,-72.126694,MESSAGEix-CA,Saskatchewan,baseline,billion US$2010/yr,Cost|Cost Nodal Net
27,0.140407,0.045009,0.063292,0.091550,0.111454,0.131452,0.363168,1.771990,-12.864151,MESSAGEix-CA,Yukon,baseline,billion US$2010/yr,Cost|Cost Nodal Net


In [541]:
report(mp= mp, scen=netzero_1, out_dir="reporting_outputs")

processing Table: Resource|Extraction
processing Table: Resource|Cumulative Extraction
processing Table: Primary Energy
PostProcess.out() got an unexpected keyword argument 'inpfilter'
processing Table: Primary Energy (substitution method)
PostProcess.out() got an unexpected keyword argument 'inpfilter'
processing Table: Final Energy
processing Table: Secondary Energy|Electricity
processing Table: Secondary Energy|Heat
processing Table: Secondary Energy
processing Table: Secondary Energy|Gases
processing Table: Secondary Energy|Solids
processing Table: Emissions|CO2
processing Table: Carbon Sequestration
processing Table: Emissions|BC
processing Table: Emissions|OC
processing Table: Emissions|CO
processing Table: Emissions|N2O
processing Table: Emissions|CH4
processing Table: Emissions|NH3
processing Table: Emissions|Sulfur
processing Table: Emissions|NOx
processing Table: Emissions|VOC
processing Table: Emissions|HFC
HFC125
No objects to concatenate
processing Table: Emissions
process

,2020,2025,2030,2035,2040,2045,2050,2055,2060,Model,Region,Scenario,Unit,Variable
1,8660.636917,8146.054912,8146.054912,8146.054912,8154.793451,10800.513484,11184.439150,8146.054912,20186.999410,MESSAGEix-CA,Alberta,netzero_1,PJ/yr,Resource|Extraction
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MESSAGEix-CA,Alberta,netzero_1,PJ/yr,Resource|Extraction|Coal
3,678.245547,163.663541,163.663541,163.663541,172.402081,287.479401,594.859532,163.663541,129.757337,MESSAGEix-CA,Alberta,netzero_1,PJ/yr,Resource|Extraction|Gas
4,678.245547,163.663541,163.663541,163.663541,172.402081,287.479401,594.859532,163.663541,129.757337,MESSAGEix-CA,Alberta,netzero_1,PJ/yr,Resource|Extraction|Gas|Conventional
5,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,MESSAGEix-CA,Alberta,netzero_1,PJ/yr,Resource|Extraction|Gas|Unconventional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,1.378249,1.335387,0.604738,1.042933,1.625089,1.135750,3.193522,3.742876,-25.006822,MESSAGEix-CA,PrinceEdwardIsland,netzero_1,billion US$2010/yr,Cost|Cost Nodal Net
23,93.468882,91.343579,60.963313,58.965245,57.609128,32.261151,38.639036,115.115316,-28.304659,MESSAGEix-CA,Quebec,netzero_1,billion US$2010/yr,Cost|Cost Nodal Net
25,10.618205,30.053304,36.311635,27.527577,25.507770,10.411413,12.240879,37.620529,-22.163093,MESSAGEix-CA,Saskatchewan,netzero_1,billion US$2010/yr,Cost|Cost Nodal Net
27,0.145979,0.340483,0.120257,0.115458,0.176123,0.113193,0.368588,0.632885,-2.624340,MESSAGEix-CA,Yukon,netzero_1,billion US$2010/yr,Cost|Cost Nodal Net


In [543]:
import pandas as pd
import pyam 
df = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/reporting_outputs/MESSAGEix-CA_baseline.xlsx')

# Convert only the negative values in the 'value' column to their absolute values
# Select only the columns from 2020 to 2060 and apply the abs() function to make values positive
df.loc[:, 2025:2060] = df.loc[:, 2025:2060].abs()
df = df.dropna(subset=['Region'])
df = df.drop_duplicates(subset=  ['Model',   'Region', 'Scenario',
           'Unit', 'Variable'])
df_pyam = pyam.IamDataFrame(df)
df =df_pyam.as_pandas()
df['value'] = df['value'].astype(float)
df['model'] = 'MESSAGEix-Canada'
df['scenario'] = 'Current Measures'
df.rename(columns={'year':'time'}, inplace=True)
df.to_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/reporting_outputs/output_pyam_baseline.csv')

In [546]:
import pandas as pd
import pyam 
df = pd.read_excel('/Users/muhammadawais/Documents/GitHub/message-ix-canada/reporting_outputs/MESSAGEix-CA_netzero_1.xlsx')

# Convert only the negative values in the 'value' column to their absolute values
# Select only the columns from 2020 to 2060 and apply the abs() function to make values positive
df.loc[:, 2025:2060] = df.loc[:, 2025:2060].abs()
df = df.dropna(subset=['Region'])
df = df.drop_duplicates(subset=  ['Model',   'Region', 'Scenario',
           'Unit', 'Variable'])
df_pyam = pyam.IamDataFrame(df)
df =df_pyam.as_pandas()
df['value'] = df['value'].astype(float)
df['model'] = 'MESSAGEix-Canada'
df['scenario'] = 'NetZero'
df.rename(columns={'year':'time'}, inplace=True)
df.to_csv('/Users/muhammadawais/Documents/GitHub/message-ix-canada/reporting_outputs/output_pyam_netzero_1.csv')

1.103378
Elapsed time = 158527.55 sec. (4316713.55 ticks, 702033 iterations)
Iteration: 702681   Dual objective     =  129070289562.081787
Iteration: 702693   Dual objective     =  129078791738.281738
Elapsed time = 158733.91 sec. (4326722.69 ticks, 702944 iterations)
Iteration: 703004   Dual objective     =  130255341080.500916
Elapsed time = 158934.04 sec. (4336740.18 ticks, 703310 iterations)
Elapsed time = 159081.62 sec. (4346984.12 ticks, 703453 iterations)
Elapsed time = 159226.02 sec. (4357347.09 ticks, 703767 iterations)
Elapsed time = 159344.46 sec. (4367353.65 ticks, 703898 iterations)
Iteration: 704044   Dual objective     =  130304922029.766098
Elapsed time = 159482.40 sec. (4381264.22 ticks, 704098 iterations)
Iteration: 704116   Dual objective     =  130304924794.930115
Elapsed time = 159643.89 sec. (4398379.69 ticks, 704162 iterations)
Iteration: 704231   Dual objective     =  130304927795.277130
Elapsed time = 159783.07 sec. (4409405.20 ticks, 704232 iterations)
Elapsed

In [169]:
netzero_1.change_first_model_year(2020)

AttributeError: 'Scenario' object has no attribute 'change_first_model_year'

In [146]:
netzero.par('bound_emission')

,node,type_emission,type_tec,type_year,value,unit
0,Canada,TCE,all,cumulative,4363.2,-


In [518]:
netzero_1 = msg.clone(msg.model, "netzero_1", keep_solution=False)

In [519]:
transfer_demands(netzero, netzero_1)

Transfer data from MESSAGEix-CA/netzero#21 variable 'DEMAND'
to MESSAGEix-CA/netzero_1#1 parameter 'demand'
702 observations


In [528]:
df= netzero_1.par('bound_emission')

In [532]:
import pandas as pd

# Define new values for each year
new_values = {
    '2025': 199.523825,
    2030: 107.426983,
    2035: 71.188808,
    2040: 45.701622,
    2045: 40,
    2050: 40,
    2055: 40,
    2060: 40
}



# Update only the values that exist in new_values
for year, new_value in new_values.items():
    df.loc[df["type_year"] == year, "value"] = new_value




In [533]:
df

,node,type_emission,type_tec,type_year,value,unit
0,Canada,TCE,all,2025,199.523825,???
1,Canada,TCE,all,2030,107.426983,???
2,Canada,TCE,all,2035,71.188808,???
3,Canada,TCE,all,2040,45.701622,???
4,Canada,TCE,all,2045,40.000000,???
5,Canada,TCE,all,2050,40.000000,???
6,Canada,TCE,all,2055,40.000000,???
7,Canada,TCE,all,2060,40.000000,???
8,Canada,TCE,all,2020,350.000000,???


In [534]:
with netzero_1.transact():
    netzero_1.add_par('bound_emission', df)

In [195]:
def assign_cumulative_netzero(scenario):
    # Define emissions data for Net-Zero scenario
    start_year = 2025
    target_year = 2050
    end_year = 2060

    initial_emission = 720  # MtCO2e in 2025
    net_zero_emission = 0  # Target emissions in 2050

    # Assuming a linear decline from 2025 to 2050
    years = list(range(start_year, end_year + 1))
    emissions_netzero = []

    # Calculate the annual decline rate for Net-Zero
    decline_rate_netzero = (net_zero_emission - initial_emission) / (target_year - start_year)

    # Calculate emissions for each year
    for year in years:
        if year <= target_year:
            emissions_netzero.append(initial_emission + (year - start_year) * decline_rate_netzero)
        else:  # Assuming zero emissions after 2050
            emissions_netzero.append(net_zero_emission)

    # Calculate cumulative emissions for Net-Zero scenario
    cumulative_budget_netzero = sum(emissions_netzero)

    # Convert MtCO2e to MtC
    cumulative_budget_netzero_mtC = cumulative_budget_netzero * 0.2727

    # Display results
    print(f'{cumulative_budget_netzero_mtC} million tonne of Carbon for Net Zero by 2050')

    import pandas as pd

    # Recreate the original DataFrame with the given trajectory
    data = {
        "type_year": list(range(2025, 2061, 5)),
        "value": [659, 549, 439, 329, 220, 110, 110, 50, 50]  # Original values in MtCO2
    }

    # Create DataFrame
    df_bound_net = pd.DataFrame(data)

    # Add required columns
    df_bound_net.insert(0, "node", "Canada")
    df_bound_net.insert(1, "type_emission", "TCE")
    df_bound_net.insert(2, "type_tec", "all")

    # Convert values from MtCO2 to MtC
    df_bound_net["value"] = df_bound_net["value"] * (12 / 44)

    with netzero_1.transact():
        df_tax = netzero_1.par('tax_emission')
        df_tax = df_tax[df_tax['type_tec'] == 'all']
        df_bound = netzero_1.par('bound_emission')
        df_bound['value'] = cumulative_budget_netzero_mtC
        # netzero_1.remove_par('tax_emission', df)
        # netzero_1.add_par('bound_emission', df)
        netzero_1.add_par('bound_emission', df_bound_net)

In [203]:
0.0116 * 1e3 *365 /8760

0.48333333333333334

In [204]:
860*0.48


412.8

In [201]:
data = {
        "type_year": list(range(2025, 2061, 5)),
        "value": [659, 549, 439, 329, 220, 110, 110,50,50]
    }

In [202]:
data

{'type_year': [2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060],
 'value': [659, 549, 439, 329, 220, 110, 110, 50, 50]}

In [197]:
df = netzero_1.par('bound_emission')

In [198]:
df

,node,type_emission,type_tec,type_year,value,unit
0,Canada,TCE,all,2020,659.0,???
1,Canada,TCE,all,2025,549.0,???
2,Canada,TCE,all,2030,439.0,???
3,Canada,TCE,all,2035,329.0,???
4,Canada,TCE,all,2040,220.0,???
5,Canada,TCE,all,2045,110.0,???
6,Canada,TCE,all,2050,50.0,???
7,Canada,TCE,all,2055,50.0,???
8,Canada,TCE,all,2060,50.0,???


In [176]:
netzero_1.remove_solution()

In [196]:
assign_cumulative_netzero(netzero_1)

2552.472 million tonne of Carbon for Net Zero by 2050


In [ ]:
caseName = netzero.model + '__' + netzero.scenario + '__v' + str(netzero.version)
netzero.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '4', "scaind": "1","writelp":"true"})

In [535]:
caseName = netzero_1.model + '__' + netzero_1.scenario + '__v' + str(netzero_1.version)
netzero_1.solve(model='MESSAGE', case=caseName, solve_options={'lpmethod': '2'})

--- Warning: The GAMS version [45.1.0] differs from the API version [24.8.3].
--- Job MESSAGE_run.gms Start 02/28/25 00:27:58 45.1.0 88bbff72 DEX-DEG x86 64bit/macOS
--- Applying:
    /Library/Frameworks/GAMS.framework/Versions/45/Resources/gmsprmun.txt
--- GAMS Parameters defined
    Input /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/MESSAGE_run.gms
    ScrDir /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/225i/
    SysDir /Library/Frameworks/GAMS.framework/Versions/45/Resources/
    LogOption 4
    --in /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/data/MsgData_MESSAGEix-CA__netzero_1__v1.gdx
    --out /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgOutput_MESSAGEix-CA__netzero_1__v1.gdx
    --iter /Users/muhammadawais/Documents/GitHub/message_ix/message_ix/model/output/MsgIterationReport_MESSAGEix-CA__netzero_1__v1.gdx
Licensee: Small MUD - 5 User License                     S220901|0002AP-GEN
  

2025-02-28 00:55:16,483 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'I' not found in gdx!
2025-02-28 00:55:16,552 ERROR at.ac.iiasa.ixmp.objects.Scenario:1691 - variable 'C' not found in gdx!


In [164]:
netzero_1.var('EMISS', {'node': 'Canada', 'emission': 'TCE', 'type_tec': 'all'})['lvl']

0     444.415570
1     370.900566
2     438.243814
3     564.960024
4     761.906945
5    1044.571742
6    1423.573916
7    2531.680741
8    3546.216774
Name: lvl, dtype: float64

,node,type_emission,type_tec,type_year,value,unit
0,Canada,TCE,all,cumulative,2552.472,-


(9360.0, 2552.472)